In [352]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [353]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [354]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [355]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
  print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [356]:
C = torch.randn((27, 2)) #for table lookup, 27 characters, 2-dimensional embedding

In [357]:
C[X].shape

torch.Size([32, 3, 2])

In [358]:
X[13, 2]

tensor(1)

In [359]:
C[X] [13, 2] # === C[1]

tensor([-1.1522, -0.2805])

In [360]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [361]:
W1 = torch.randn((6, 100)) # 6 inputs, 100 outputs
b1 = torch.randn(100) # 100 outputs


# W1 is a weight matrix that transforms the 6-dimensional input into a 
# 100-dimensional output. Each row of W1 corresponds to a different input feature, 
# and each column corresponds to 
# a different output feature. The bias vector b1 is added to
# the output of the linear transformation to allow for shifting the activation function.

In [362]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # matrix multiplication

W1 is a weight matrix that transforms the 6-dimensional input into a 100-dimensional output. 
Each row of W1 corresponds to a different input feature, and each column corresponds to a different output feature. The bias 
vector b1 is added to the output of the linear transformation to allow for shifting the activation function.

    
 we use view(-1, 6) here because we want to flatten the embedding tensor into
a 2D tensor with shape (batch_size, 6) so that we can perform matrix multiplication with W1. The -1 in 
view(-1, 6) means that the batch size will be inferred automatically based on the size of the input tensor.

In [363]:
W2 = torch.randn((100, 27)) 
b2 = torch.randn(27) 

In [364]:
logits = h @ W2 + b2

In [365]:
logits

tensor([[ 3.9543e+00, -6.4110e-01,  2.0216e+00, -1.0988e+01,  1.7774e+01,
          1.4141e+01,  1.6104e+01,  5.2294e+00,  1.4531e+00,  1.9907e+00,
          7.0539e+00, -1.5063e+00,  4.5856e+00,  1.8058e+01,  1.4778e+01,
          5.6286e+00, -4.3983e+00, -5.3139e+00, -5.7709e+00,  2.2643e+00,
         -1.0821e+00, -6.1037e+00,  2.2971e+01,  1.0107e+01,  2.0314e+00,
         -2.0019e+01, -1.3725e+01],
        [-4.8453e+00,  5.4327e-01,  1.7525e+00, -2.2686e+01,  1.0349e+01,
          2.3894e+00,  1.1606e+01,  1.6390e+00,  6.0952e-01,  7.4730e+00,
          7.2347e+00,  1.7341e+00,  1.7395e+00,  9.9830e+00,  2.1598e+01,
          1.2572e+01,  3.4223e-01,  2.6405e+00, -2.0370e+00,  2.7296e+00,
          4.3807e+00, -9.3348e-01,  1.5688e+01,  1.1158e+00, -7.5693e+00,
         -1.2209e+01, -3.6638e+00],
        [-3.0959e+00, -7.1568e-01, -9.7184e+00, -2.1556e+01,  2.6892e+00,
         -1.0041e+00,  6.3620e+00,  7.6642e+00,  1.2127e+01, -3.0600e+00,
          5.0510e+00, -2.8389e+00, -3.39

In [366]:
counts = logits.exp() # counts  

In [367]:
prob = counts / counts.sum(1, keepdims=True) 

In [368]:
prob.shape

torch.Size([32, 27])

In [369]:
loss = -prob[torch.arange(32), Y].log().mean() 

loss
# NLL loss Nll is the negative log likelihood loss, which is commonly used for multi-class classification problems. 
# It measures how well the predicted probabilities match the true labels. 
# The loss is computed by taking the negative logarithm of the predicted probability for the true class and
# averaging it over all samples in the batch.

tensor(17.1618)

In [370]:
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

-------------------------------------------CLEANING CODE-------------------------------------------------------

In [371]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [372]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [ ]:
emb = C[X] # embed the characters into vectors
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # hidden layer   
logits = h @ W2 + b2
# counts = logits.exp() # counts, equivalent to N
# probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
# loss = -prob[torch.arange(32), Y].log().mean() 

F.cross_entropy(logits, Y) # computes the same
loss


tensor(17.1618)